In [31]:
# DSI doctoral certificate
# visualization assignment 3
# June 16, 2026, Robert Lu

import numpy as np
import pandas as pd
import requests
import pprint
from datetime import datetime
import os
from pathlib import Path

date = datetime.today().strftime("%Y-%m-%d")
print(date)

cwd = os.getcwd()
print(cwd)

2026-06-16
/Users/robertlu/Documents/Biochemistry/DSI/visualization/assignment_3_files


In [ ]:
# below is a function which downloads some data from the City of Toronto
# and saves the files as a csv. 
# based on example code here https://open.toronto.ca/dataset/ttc-bus-delay-data/
# and in other dataset pages

def get_toronto_data_as_csv (id="ttc-bus-delay-data", 
    base_url="https://ckan0.cf.opendata.inter.prod-toronto.ca", save_folder_name = "saved_data"):

    """
    Downloads active datasets from the city of toronto

    Find datasets here: https://open.toronto.ca/catalogue/

    The function in its current form downloads ttc delay data as a csv

    Arguments:
        id: string, name of the dataset
        base_url: string, where the data is located. may change based on dataset
        save_folder_name: string, name of folder to save data within current directory
    
    Function code based on example code here https://open.toronto.ca/dataset/ttc-bus-delay-data/

    """

    # The dataset exists as a package
    # apparently "datasets" and "package" is the same thing, package is an older name
    # https://docs.ckan.org/en/latest/user-guide.html
    url = base_url + "/api/3/action/package_show" 
    params = {"id": id}
    package = requests.get(url, params = params).json()

    # To get resource data:
    # within the package (dataset) there are specific datasets, and those datasets have resources
    # the data is accessed by "result", and individual datasets are "resources"
    # like there's some other random info in "result" so we don't need that
    for idx, resource in enumerate(package["result"]["resources"]):

        # for datastore_active resources:
        # check if the dataset is active? For the ttc delay data this is recent data 
        # which can be downloaded as a csv
        if resource["datastore_active"]:
        
            # To get all records in CSV format:
            url = base_url + "/datastore/dump/" + resource["id"]
            resource_dump_data = requests.get(url).text
            #print(resource_dump_data)

            # try to create a separate folder to save the data
            cwd = os.getcwd()
            save_loc = cwd + "/" + save_folder_name
            save_path = Path(cwd, save_folder_name)
            save_path.mkdir(parents=True, exist_ok=True)
            
            # this is here because the ttc delay data has active datasets called "Code Descriptions"
            # which just describes what some codes in their data mean
            # I think the naming is the same for all delay datasets, so create a unique name
            # for each of the code descriptions
            date = datetime.today().strftime("%Y-%m-%d")
            resource_name = resource["name"].replace(" ", "_")
            if resource_name == "Code_Descriptions":
                csv_name = save_loc + "/" + date + "_" + params["id"] + "_" + resource_name + ".csv"
            else:
                csv_name = save_loc + "/" + date + "_" + resource_name + ".csv"

            # save the data as a csv file
            with open(csv_name, "w", encoding="utf-8") as file:
                file.write(resource_dump_data)

            # other demo code below which I kept just in case 
            
            # To selectively pull records and attribute-level metadata:
            url = base_url + "/api/3/action/datastore_search"
            p = { "id": resource["id"] }
            resource_search_data = requests.get(url, params = p).json()["result"]
            #print(resource_search_data)
            # This API call has many parameters. They're documented here:
            # https://docs.ckan.org/en/latest/maintaining/datastore.html
    
        # To get metadata for non datastore_active resources:
        if not resource["datastore_active"]:
            url = base_url + "/api/3/action/resource_show?id=" + resource["id"]
            resource_metadata = requests.get(url).json()
            #print(resource_metadata)
            # From here, you can use the "url" attribute to download this file

In [ ]:
# download the relevant data

get_toronto_data_as_csv("ttc-bus-delay-data")
get_toronto_data_as_csv("ttc-streetcar-delay-data")
get_toronto_data_as_csv("ttc-subway-delay-data")
get_toronto_data_as_csv("ttc-lrt-delay-data")

# takes about 12 sec to run